In [0]:
eilerspath='development_042_silver_sandbox.demand_forecast.eilers_landbased_performance_clean_sl'
mappingpath='development_042_silver_sandbox.demand_forecast.eilers_to_gdna_mapping_silver'
dest='development_042_silver_sandbox.demand_forecast.eilers_group_historical_time_series'

In [0]:
eilers=spark.read.table(eilerspath)
mapping_df = spark.read.table(mappingpath)
display(eilers)

In [0]:
from pyspark.sql import functions as F

joined_df = eilers.join(
    mapping_df,
    eilers.game_name== F.lower(mapping_df.bp_theme_name),
    "inner"
)

display(joined_df)

In [0]:
from pyspark.sql import functions as F

# Clean game_name: remove dashes, extra spaces, and trim
eilers = eilers.withColumn(
    "game_name",
    F.trim(F.regexp_replace(F.col("game_name"), r"[\-\–\—]", " "))
).withColumn(
    "game_name",
    F.trim(F.regexp_replace(F.col("game_name"), r"\s+", " "))
).withColumn(
    "game_name",
    F.lower(F.col("game_name"))
)

# Filter for U.S. & Canada region
eilers_us = eilers.filter(F.col("region_name") == "U.S. & Canada")

# Define grouping columns
group_cols = ["game_name", "yearmonth", "own_status"]

mode_cols = [
    "game_classification",
    "game_category",
    "parent_game",
    "parent_own_status",
    "supplier",
]
# Columns to sum
sum_cols = ["no_of_slots", "casinos"]

# Columns to weighted-average by no_of_slots
weighted_avg_cols = [
    "avg_average_bet",
    "avg_days_game_theme",
    "avg_days_terminal",
    "avg_games_played_index",
    "avg_coin_in_index_vs_house",
    "avg_theo_net_win_index_vs_house",
    "avg_coin_in_vs_zone",
    "avg_theo_net_win_index_vs_zone",
    "average_bet_index_same_denom",
    "average_bet_index_same_type_same_denom",
    "coin_in_vs_house_same_type",
    "theo_win_vs_house_same_type_same_denom",
    "utilization_index_same_type_same_denom",
]

# Build aggregation expressions
agg_exprs = [F.sum(F.col(c)).alias(c) for c in sum_cols]

# Weighted average: sum(col * no_of_slots) / sum(no_of_slots)
agg_exprs += [
    (F.sum(F.col(c) * F.col("no_of_slots")) / F.sum(F.col("no_of_slots"))).alias(c)
    for c in weighted_avg_cols
]

# Min game_release_date per group
agg_exprs.append(F.min(F.col("game_release_date")).alias("game_release_date"))

# Apply grouping and aggregation
from pyspark.sql.window import Window

# Aggregate numeric metrics at the required grain.
eilers_grouped = eilers_us.groupBy(*group_cols).agg(*agg_exprs)

# Add the most frequent non-null value for each descriptive column.
# Ties are resolved alphabetically for consistent results.
for column_name in mode_cols:
    mode_window = Window.partitionBy(*group_cols).orderBy(
        F.desc("_frequency"),
        F.col(column_name).asc(),
    )

    modes = (
        eilers_us
        .where(F.col(column_name).isNotNull())
        .groupBy(*(group_cols + [column_name]))
        .agg(F.count("*").alias("_frequency"))
        .withColumn("_rank", F.row_number().over(mode_window))
        .where(F.col("_rank") == 1)
        .select(*group_cols, column_name)
    )

    join_condition = [
        F.col(f"g.{key}").eqNullSafe(F.col(f"m.{key}"))
        for key in group_cols
    ]

    eilers_grouped = (
        eilers_grouped.alias("g")
        .join(modes.alias("m"), join_condition, "left")
        .select("g.*", F.col(f"m.{column_name}").alias(column_name))
    )
# Convert game_release_date to months_since_release and drop the original column
eilers_grouped = eilers_grouped.withColumn(
    "months_since_release",
    F.months_between(
        F.col("yearmonth"),
        F.col("game_release_date")
    ).cast("int")
)#.drop("game_release_date")

# Replace nulls with column means for the aggregated metric columns
mean_values = eilers_grouped.select(
    [F.mean(F.col(c)).alias(c) for c in sum_cols + weighted_avg_cols]
).first().asDict()

eilers_grouped = eilers_grouped.fillna(mean_values)

display(eilers_grouped)

In [0]:
from pyspark.sql import functions as F

# Expected grain: one row per game, month, and own status.
key_cols = ["game_name", "yearmonth", "own_status"]
extra_cols = [c for c in group_cols if c not in key_cols]

# Count null as a distinct value when checking attribute differences.
checks = eilers_grouped.groupBy(*key_cols).agg(
    F.count("*").alias("rows_per_period"),
    *[
        F.countDistinct(F.struct(F.col(c))).alias(f"{c}_distinct")
        for c in extra_cols
    ],
).cache()

duplicates = checks.where(F.col("rows_per_period") > 1)

total_periods = checks.count()
duplicate_periods = duplicates.count()

print(f"Game/month/own-status combinations: {total_periods:,}")
print(f"Combinations with multiple rows: {duplicate_periods:,}")

if duplicate_periods == 0:
    print("PASS: One row per game, month, and own status.")
else:
    print("FAIL: Additional grouping attributes split some periods.")

    display(duplicates.orderBy(F.desc("rows_per_period")))

    # Null-safe join also includes duplicate keys containing nulls.
    detail_condition = (
        F.col("g.game_name").eqNullSafe(F.col("d.game_name"))
        & F.col("g.yearmonth").eqNullSafe(F.col("d.yearmonth"))
        & F.col("g.own_status").eqNullSafe(F.col("d.own_status"))
    )

    display(
        eilers_grouped.alias("g")
        .join(
            duplicates.select(*key_cols).alias("d"),
            detail_condition,
            "left_semi",
        )
        .orderBy(*key_cols)
    )

checks.unpersist()

In [0]:
eilers_grouped.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(dest)